In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import time
import itertools

In [2]:
%matplotlib inline

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.family']='Malgun Gothic'

# 부산 비교하기

In [52]:
df_23 = pd.read_csv('../../DB/보리_전부_부산예측2023.csv', encoding='CP949')
print(f"데이터프레임 shape: {df_23.shape}")
df_23.head(3)

데이터프레임 shape: (16, 7)


,행정구역,연도,총인구수,혼합배출_후_재활용량,first,second,third
0,부산_강서구,2023,142396,11805.7,10470.784,11137.885,10192.602
1,부산_금정구,2023,215590,24013.1,8918.680,9889.570,9225.789
2,부산_기장군,2023,178729,13154.8,7028.226,8365.985,7466.829


In [53]:
df_22 = pd.read_csv('../../DB/보리_부산비교용2022.csv', encoding='CP949')
df_22 = df_22[['행정구역','연도','총인구수','혼합배출_후_재활용량']]
print(f"데이터프레임 shape: {df_22.shape}")
df_22.head(3)

데이터프레임 shape: (16, 4)


,행정구역,연도,총인구수,혼합배출_후_재활용량
0,부산_강서구,2022,143207,18942.6
1,부산_금정구,2022,221256,19162.7
2,부산_기장군,2022,178614,7995.3


In [16]:
# Test 값 정리
predict_list = pd.DataFrame({'first': {'MSE': 127905910.11622496,
  'RMSE': np.float64(11309.54950987107),
  'R^2': 0.532922673355962,
  'MAE': 6221.976114035087},
 'second': {'MSE': 126484539.21180522,
  'RMSE': np.float64(11246.53454232926),
  'R^2': 0.5381131303223581,
  'MAE': 6277.309736842106},
 'third': {'MSE': 123968346.36350507,
  'RMSE': np.float64(11134.107344709097),
  'R^2': 0.5473015769534555,
  'MAE': 5886.821052631578}})
# 부산 값 정리
busan_list = pd.DataFrame({'first': {'MSE': 62689390.399004646,
  'RMSE': np.float64(7917.663190550899),
  'R^2': 0.06324374263693655,
  'MAE': 6467.360874999999},
 'second': {'MSE': 66712170.86464327,
  'RMSE': np.float64(8167.7518855951585),
  'R^2': 0.003132059476515958,
  'MAE': 6777.627},
 'third': {'MSE': 68851316.38341592,
  'RMSE': np.float64(8297.669334422524),
  'R^2': -0.028832806306454906,
  'MAE': 6834.354812499999}})

In [18]:
print("# Test 값 정리")
predict_list.T

# Test 값 정리


,MSE,RMSE,R^2,MAE
first,1.279059e+08,11309.549510,0.532923,6221.976114
second,1.264845e+08,11246.534542,0.538113,6277.309737
third,1.239683e+08,11134.107345,0.547302,5886.821053


1. 로그 취한 값
[train-valid 성능 테스트]
R2 (Validation): 0.6325
MSE (Validation): 0.7238

[2023년 테스트 성능 평가]
2023년 로그 기준 R2: 0.3589
2023년 로그 기준 MSE: 1.0475
2023년 실제값 기준 R2: 0.3002
2023년 실제값 기준 MSE: 175246016.8329

[2023년 부산 지역 테스트 성능 평가]
23년 부산 로그 기준 R2: -1.8398
23년 부산 기준 MSE: 0.5426
23년 부산 실제값 기준 R2: -0.3879
23년 부산 실제값 기준 MSE: 92882371.6528

2. 로그 취하지 않은 값

변수: [총인구수, 아파트비율, 다세대주택비율, 2인가구_비율, 순이동]
[train-valid 성능 테스트]
R2 (Validation): 0.4930
MSE (Validation): 25845041.6078

[2023년 테스트 성능 평가]
2023년 실제값 기준 R2: 0.3569
2023년 실제값 기준 MSE: 161045151.8089

[2023년 부산 지역 테스트 성능 평가]
23년 부산 실제값 기준 R2: 0.0101
23년 부산 실제값 기준 MSE: 66242807.6850


In [17]:
print("# 부산 값 정리")
busan_list.T

# 부산 값 정리


,MSE,RMSE,R^2,MAE
first,6.268939e+07,7917.663191,0.063244,6467.360875
second,6.671217e+07,8167.751886,0.003132,6777.627000
third,6.885132e+07,8297.669334,-0.028833,6834.354812


# first

In [24]:
df_23

Index(['행정구역', '연도', '총인구수', '혼합배출_후_재활용량', 'first', 'second', 'third'], dtype='object')

## 2023년만

In [54]:
result1 = df_23[['행정구역','first']].sort_values(by='first',ascending=False)
result1

,행정구역,first
6,부산_부산진구,24339.463
9,부산_사하구,19433.512
15,부산_해운대구,18258.229
5,부산_동래구,16353.151
7,부산_북구,14006.110
3,부산_남구,12471.135
8,부산_사상구,11605.771
12,부산_연제구,11059.517
0,부산_강서구,10470.784
1,부산_금정구,8918.680


In [55]:
result1['23년만'] = result1['first'].rank(ascending=False).astype(int)

In [56]:
result1.head(3)

,행정구역,first,23년만
6,부산_부산진구,24339.463,1
9,부산_사하구,19433.512,2
15,부산_해운대구,18258.229,3


## 부산2023/인구수

In [57]:
df_23['1인당_혼합배출_first'] = df_23['first']/df_23['총인구수']
result2 = df_23[['행정구역','1인당_혼합배출_first']].sort_values(by='1인당_혼합배출_first',ascending=False)
result2

,행정구역,1인당_혼합배출_first
14,부산_중구,0.230132
0,부산_강서구,0.073533
4,부산_동구,0.068426
6,부산_부산진구,0.067702
9,부산_사하구,0.065250
5,부산_동래구,0.060385
8,부산_사상구,0.057190
12,부산_연제구,0.053748
13,부산_영도구,0.053643
7,부산_북구,0.051193


In [58]:
result2['1인당'] = result2['1인당_혼합배출_first'].rank(ascending=False).astype(int)

# 부산2023- 부산2022

In [59]:
merge = pd.merge(df_23[['행정구역','first']], df_22[['행정구역','혼합배출_후_재활용량']], on='행정구역')
merge['증감'] = merge['first'] - merge['혼합배출_후_재활용량'] 
merge['증감률'] = (merge['first'] - merge['혼합배출_후_재활용량'])/merge['혼합배출_후_재활용량'] * 100
result3 = merge[['행정구역', '증감', '증감률']].sort_values(by='증감', ascending=False)
result3

,행정구역,증감,증감률
7,부산_북구,1626.210,13.135890
8,부산_사상구,1481.271,14.630560
13,부산_영도구,-893.789,-13.523202
2,부산_기장군,-967.074,-12.095531
11,부산_수영구,-1604.113,-19.836190
14,부산_중구,-2179.528,-19.693937
4,부산_동구,-2855.587,-32.219919
15,부산_해운대구,-5092.771,-21.809648
9,부산_사하구,-5694.788,-22.662846
10,부산_서구,-5862.833,-60.220562


In [60]:
result3['증감'] = result3['증감'].rank(ascending=False).astype(int)

In [62]:
result4 = merge[['행정구역', '증감', '증감률']].sort_values(by='증감률', ascending=False)
result4

,행정구역,증감,증감률
8,부산_사상구,1481.271,14.630560
7,부산_북구,1626.210,13.135890
2,부산_기장군,-967.074,-12.095531
13,부산_영도구,-893.789,-13.523202
14,부산_중구,-2179.528,-19.693937
11,부산_수영구,-1604.113,-19.836190
15,부산_해운대구,-5092.771,-21.809648
9,부산_사하구,-5694.788,-22.662846
6,부산_부산진구,-9115.237,-27.246506
5,부산_동래구,-7678.549,-31.951751


In [63]:
result4['증감률'] = result4['증감률'].rank(ascending=False).astype(int)

# 부산23/인구수 - 부산22/인구수

In [64]:
df_22['1인당_혼합배출_22'] = df_22['혼합배출_후_재활용량']/df_22['총인구수']

In [65]:
merge = pd.merge(df_23[['행정구역','1인당_혼합배출_first']], df_22[['행정구역','1인당_혼합배출_22']], on='행정구역')
merge['1인당_증감'] = merge['1인당_혼합배출_first'] - merge['1인당_혼합배출_22'] 
merge['1인당_증감률'] = (merge['1인당_혼합배출_first'] - merge['1인당_혼합배출_22'] )/merge['1인당_혼합배출_22']  * 100
result5 = merge[['행정구역', '1인당_증감', '1인당_증감률']].sort_values(by='1인당_증감', ascending=False)
result5

,행정구역,1인당_증감,1인당_증감률
8,부산_사상구,0.007509,15.114655
7,부산_북구,0.006798,15.311389
2,부산_기장군,-0.005440,-12.152092
13,부산_영도구,-0.007466,-12.218113
11,부산_수영구,-0.009115,-19.703899
15,부산_해운대구,-0.012381,-20.507257
9,부산_사하구,-0.017960,-21.583666
3,부산_남구,-0.025416,-34.125015
6,부산_부산진구,-0.026294,-27.973216
5,부산_동래구,-0.027570,-31.345934


In [66]:
result5['1인당_증감'] = result5['1인당_증감'].rank(ascending=False).astype(int)

In [67]:
result6 = merge[['행정구역', '1인당_증감', '1인당_증감률']].sort_values(by='1인당_증감률', ascending=False)
result6

,행정구역,1인당_증감,1인당_증감률
7,부산_북구,0.006798,15.311389
8,부산_사상구,0.007509,15.114655
2,부산_기장군,-0.005440,-12.152092
13,부산_영도구,-0.007466,-12.218113
14,부산_중구,-0.048711,-17.468931
11,부산_수영구,-0.009115,-19.703899
15,부산_해운대구,-0.012381,-20.507257
9,부산_사하구,-0.017960,-21.583666
6,부산_부산진구,-0.026294,-27.973216
5,부산_동래구,-0.027570,-31.345934


In [68]:
result6['1인당_증감률'] = result6['1인당_증감률'].rank(ascending=False).astype(int)

# 혼합배출(보리) / 종량제(민주)

In [70]:
total_23 = pd.read_csv('../../DB/부산예측2023.csv', encoding='CP949')
print(f"데이터프레임 shape: {total_23.shape}")
total_23.head(3)

데이터프레임 shape: (16, 6)


,행정구역,연도,총인구수,발생량,새로운_발생량,오차(실제값기준)
0,부산_강서구,2023,142396,17160.4,21781.747,-4621.347
1,부산_금정구,2023,215590,32190.5,29053.355,3137.145
2,부산_기장군,2023,178729,30226.6,27378.295,2848.305


In [71]:
merge = pd.merge(df_23[['행정구역','first']], total_23[['행정구역','새로운_발생량']], on='행정구역')
merge['혼합배출비율'] = merge['first']/merge['새로운_발생량'] * 100
result7 = merge[['행정구역', '혼합배출비율']].sort_values(by='혼합배출비율', ascending=False)
result7

,행정구역,혼합배출비율
5,부산_동래구,57.086045
14,부산_중구,55.683204
7,부산_북구,52.057862
8,부산_사상구,51.428307
6,부산_부산진구,50.042242
9,부산_사하구,49.519494
0,부산_강서구,48.071369
13,부산_영도구,46.047778
12,부산_연제구,45.219503
4,부산_동구,44.322904


In [72]:
result7['혼합배출비율'] = result7['혼합배출비율'].rank(ascending=False).astype(int)

# 순위 모으기

In [79]:
merged_df = result1[['행정구역', '23년만']]

# 나머지 데이터프레임을 순차적으로 병합
merged_df = pd.merge(merged_df, result2[['행정구역', '1인당']], on='행정구역', how='outer')
merged_df = pd.merge(merged_df, result3[['행정구역', '증감']], on='행정구역', how='outer')
merged_df = pd.merge(merged_df, result4[['행정구역', '증감률']], on='행정구역', how='outer')
merged_df = pd.merge(merged_df, result5[['행정구역', '1인당_증감']], on='행정구역', how='outer')
merged_df = pd.merge(merged_df, result6[['행정구역', '1인당_증감률']], on='행정구역', how='outer')
merged_df = pd.merge(merged_df, result7[['행정구역', '혼합배출비율']], on='행정구역', how='outer')

merged_df

,행정구역,23년만,1인당,증감,증감률,1인당_증감,1인당_증감률,혼합배출비율
0,부산_강서구,9,2,14,14,16,14,7
1,부산_금정구,10,13,16,15,13,15,14
2,부산_기장군,12,14,4,3,3,3,15
3,부산_남구,6,11,11,12,8,12,11
4,부산_동구,14,3,7,11,11,11,10
5,부산_동래구,4,6,12,10,10,10,1
6,부산_부산진구,1,4,15,9,9,9,5
7,부산_북구,5,10,1,2,2,1,3
8,부산_사상구,7,7,2,1,1,2,4
9,부산_사하구,2,5,9,8,7,8,6


In [80]:
# 순위 합산
merged_df['합산_순위'] = merged_df['23년만'] + merged_df['1인당'] + merged_df['증감'] + merged_df['증감률'] + merged_df['1인당_증감'] + merged_df['1인당_증감률']+ merged_df['혼합배출비율']
merged_df.sort_values(by='합산_순위')

,행정구역,23년만,1인당,증감,증감률,1인당_증감,1인당_증감률,혼합배출비율,합산_순위
7,부산_북구,5,10,1,2,2,1,3,24
8,부산_사상구,7,7,2,1,1,2,4,24
14,부산_중구,11,1,6,5,14,5,2,44
9,부산_사하구,2,5,9,8,7,8,6,45
13,부산_영도구,15,9,3,4,4,4,8,47
6,부산_부산진구,1,4,15,9,9,9,5,52
5,부산_동래구,4,6,12,10,10,10,1,53
2,부산_기장군,12,14,4,3,3,3,15,54
15,부산_해운대구,3,12,8,7,6,7,12,55
11,부산_수영구,13,16,5,6,5,6,16,67


In [82]:
# 순위 합산
merged_df2=merged_df[['행정구역','23년만','1인당','혼합배출비율']]
merged_df2['합산_순위'] = merged_df2['23년만'] + merged_df2['1인당'] + merged_df2['혼합배출비율']
merged_df2.sort_values(by='합산_순위')

C:\Users\nammi\AppData\Local\Temp\ipykernel_15804\4241307802.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df2['합산_순위'] = merged_df2['23년만'] + merged_df2['1인당'] + merged_df2['혼합배출비율']


,행정구역,23년만,1인당,혼합배출비율,합산_순위
6,부산_부산진구,1,4,5,10
5,부산_동래구,4,6,1,11
9,부산_사하구,2,5,6,13
14,부산_중구,11,1,2,14
0,부산_강서구,9,2,7,18
8,부산_사상구,7,7,4,18
7,부산_북구,5,10,3,18
12,부산_연제구,8,8,9,25
15,부산_해운대구,3,12,12,27
4,부산_동구,14,3,10,27


# second